In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

import warnings
warnings.filterwarnings('ignore')

In [2]:
def enrich_and_clean_retro(base_df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищает ретро-датасет от обычного старого автохлама и генерирует бизнес-фичи.

    Args:
        base_df: Исходный датафрейм с ретро-машинами.

    Returns:
        pd.DataFrame: Очищенный и обогащенный датафрейм.
    """
    df_new = base_df.copy()

    domestic_and_cheap_asian = [
        'LADA (ВАЗ)', 'УАЗ', 'ГАЗ', 'Москвич', 'ИЖ', 'Daewoo', 'Chevrolet', 'ZAZ'
    ]

    # Маска мусора: машина от 1990 до 2000 года, из дешевых брендов, и стоит меньше 300к
    beater_mask = (
        (df_new['year'] >= 1990) &
        (df_new['brand'].isin(domestic_and_cheap_asian)) &
        (df_new['price'] < 300_000)
    )

    cars_before = len(df_new)
    df_new = df_new[~beater_mask]
    cars_after = len(df_new)
    print(f"Вычищено дешевого автохлама 90-х: удалено {cars_before - cars_after} машин.")

    bins_era = [1900, 1980, 1990, 2001]
    labels_era = ['Антиквариат (<1980)', 'Классика (1980-1990)', 'Янгтаймеры (1990-2000)']
    df_new['retro_era'] = pd.cut(df_new['year'], bins=bins_era, labels=labels_era, right=False)

    bins_tier = [0, 500_000, 1_500_000, 5_000_000, np.inf]
    labels_tier = ['Проект (<500к)', 'Энтузиаст (500к-1.5м)', 'Коллекция (1.5м-5м)', 'Музей (>5м)']
    df_new['invest_tier'] = pd.cut(df_new['price'], bins=bins_tier, labels=labels_tier, right=False)

    # Машина до 2000 года с пробегом меньше 50к - это огромная редкость
    df_new['is_time_capsule'] = np.where(df_new['mileage'] < 50_000, 'Капсула (<50к)', 'С пробегом (>50к)')
    df_new['is_time_capsule'] = df_new['is_time_capsule'].astype('category')

    return df_new


def run_retro_cleaning_pipeline(
    raw_base_path: str = 'data/raw/liquidity_base.csv',
    raw_cnt_path: str = 'data/raw/liquidity_cnt.csv',
    clean_retro_base_path: str = 'data/processed/retro_base.csv',
    clean_retro_cnt_path: str = 'data/processed/retro_cnt.csv',
    max_retro_year: int = 2000,
    min_retro_price: int = 100_000,
    bot_traffic_limit: int = 150
) -> None:
    """
    Пайплайн очистки для эксклюзивных ретро-автомобилей.
    Никаких фильтров по оверпрайсу и скрученному пробегу — только хардкор.
    """
    print("Загрузка сырых данных для Ретро-анализа...")
    base_df = pd.read_csv(raw_base_path)
    cnt_df = pd.read_csv(raw_cnt_path)

    # Защита от пробелов в колонках
    base_df.columns = base_df.columns.str.strip()
    cnt_df.columns = cnt_df.columns.str.strip()

    # Строго старые и строго дорогие
    retro_mask = (base_df['year'] <= max_retro_year) & (base_df['price'] >= min_retro_price)
    base_df = base_df[retro_mask].copy()
    print(f"Найдено машин до 2000г от 100к: {base_df.shape[0]} шт.")

    # Применяем бизнес-зачистку и новые фичи
    base_df = enrich_and_clean_retro(base_df)

    # Сносим строки с микро-пропусками
    base_df = base_df.dropna(subset=['modification'])

    # Заполняем пропуски
    base_df['rating'] = base_df['rating'].fillna(-1.0)
    base_df['reviews'] = base_df['reviews'].fillna(0)
    base_df['equipment'] = base_df['equipment'].fillna('Не указано')
    base_df['removal_reason'] = base_df['removal_reason'].fillna('Неизвестно')

    imv_fill = base_df.groupby(['brand', 'model', 'year'])['price'].transform('median')
    base_df['imv'] = base_df['imv'].fillna(imv_fill).fillna(base_df['price'])

    base_df['delta_p'] = (base_df['price'] - base_df['imv']) / base_df['imv']

    # Синхронизация логов с очищенной базой
    valid_ids = base_df['id'].unique()
    cnt_df = cnt_df[cnt_df['id'].isin(valid_ids)]

    # Ботов все равно отрезаем
    cnt_df['cnt_contacts'] = np.where(
        cnt_df['cnt_contacts'] > bot_traffic_limit,
        bot_traffic_limit,
        cnt_df['cnt_contacts']
    )

    print(f"Итоговый размер элитного ретро-парка: {base_df.shape[0]} шт.")

    print("Сохранение ретро-базы...")
    base_df.to_csv(clean_retro_base_path, index=False)
    cnt_df.to_csv(clean_retro_cnt_path, index=False)
    print("Готово. Ретро-сегмент готов к исследованию.")

if __name__ == "__main__":
    run_retro_cleaning_pipeline()

Загрузка сырых данных для Ретро-анализа...
Найдено машин до 2000г от 100к: 29672 шт.
Вычищено дешевого автохлама 90-х: удалено 1667 машин.
Итоговый размер элитного ретро-парка: 27933 шт.
Сохранение ретро-базы...
Готово. Ретро-сегмент готов к исследованию.


In [3]:
def add_geo_zones(df: pd.DataFrame) -> pd.DataFrame:
    """Возвращаем нашу топовую разбивку на 12 макро-регионов."""
    df_new = df.copy()

    moscow_city = ['Москва']
    moscow_obl = ['Московская область']
    spb_city = ['Санкт-Петербург']
    len_obl = ['Ленинградская область']
    south = ['Краснодарский край', 'Ростовская область', 'Волгоградская область', 'Астраханская область', 'Республика Адыгея', 'Республика Калмыкия']
    caucasus = ['Республика Дагестан', 'Ставропольский край', 'Чеченская Республика', 'Республика Северная Осетия — Алания', 'Кабардино-Балкарская Республика', 'Карачаево-Черкесская Республика', 'Республика Ингушетия']
    volga = ['Республика Татарстан', 'Самарская область', 'Нижегородская область', 'Республика Башкортостан', 'Пермский край', 'Саратовская область', 'Оренбургская область', 'Удмуртская Республика', 'Пензенская область', 'Кировская область', 'Чувашская Республика', 'Ульяновская область']
    ural = ['Свердловская область', 'Челябинская область', 'Тюменская область', 'Ханты-Мансийский АО', 'Ямало-Ненецкий АО', 'Курганская область']
    siberia = ['Новосибирская область', 'Красноярский край', 'Кемеровская область', 'Иркутская область', 'Алтайский край', 'Омская область', 'Томская область']
    far_east = ['Приморский край', 'Хабаровский край', 'Амурская область', 'Забайкальский край', 'Республика Саха (Якутия)', 'Сахалинская область', 'Камчатский край']
    center = ['Воронежская область', 'Белгородская область', 'Тульская область', 'Владимирская область', 'Курская область', 'Тверская область', 'Ярославская область', 'Рязанская область', 'Липецкая область', 'Ивановская область', 'Калужская область', 'Смоленская область', 'Республика Крым']

    geo_conditions = [
        df_new['region'].isin(moscow_city), df_new['region'].isin(moscow_obl),
        df_new['region'].isin(spb_city), df_new['region'].isin(len_obl),
        df_new['region'].isin(south), df_new['region'].isin(caucasus),
        df_new['region'].isin(volga), df_new['region'].isin(ural),
        df_new['region'].isin(siberia), df_new['region'].isin(far_east),
        df_new['region'].isin(center)
    ]
    geo_choices = ['Москва (Город)', 'Московская область', 'Питер (Город)', 'Ленинградская область', 'Юг РФ', 'Северный Кавказ', 'Поволжье', 'Урал', 'Сибирь', 'Дальний Восток', 'Центр РФ']
    df_new['geo_zone'] = np.select(geo_conditions, geo_choices, default='Север и СЗФО')

    return df_new

def run_retro_grid_search(df: pd.DataFrame, n_factors: int = 3, min_cars: int = 50):
    """Специальный сканер для поиска эксклюзивных ниш (порог входа снижен)."""
    print(f"Запуск Ретро-сканера гипотез (комбинации по {n_factors} факторам, минимум {min_cars} машин)...\n")

    total_cars = len(df)
    # Ищем инсайты на стыке Эпохи, Цены (Тира), Состояния и Географии
    features = ['geo_zone', 'retro_era', 'invest_tier', 'is_time_capsule']
    combinations = list(itertools.combinations(features, n_factors))
    results = []

    for combo in combinations:
        combo_list = list(combo)
        grouped = df.groupby(combo_list, observed=True).agg(
            median_contacts=('total_contacts', 'median'),
            cars_count=('id', 'count')
        ).reset_index()

        valid_groups = grouped[grouped['cars_count'] >= min_cars].copy()

        if not valid_groups.empty:
            valid_groups['segment_name'] = valid_groups[combo_list].astype(str).agg(' | '.join, axis=1)
            valid_groups['market_share_%'] = (valid_groups['cars_count'] / total_cars) * 100
            results.append(valid_groups[['segment_name', 'median_contacts', 'cars_count', 'market_share_%']])

    final_df = pd.concat(results, ignore_index=True).sort_values(by='median_contacts', ascending=False)

    print("ТОП-5 ЭКСКЛЮЗИВНЫХ АЛМАЗОВ (Высокий спрос):")
    display(final_df.head(5).round({'market_share_%': 2}))

    print("\nТОП-5 МУЗЕЙНЫХ ВИСЯКОВ (Долгая продажа):")
    display(final_df.tail(5).round({'market_share_%': 2}))

    return final_df

if __name__ == "__main__":
    print("Собираем Ретро-витрину...")
    retro_base = pd.read_csv('data/processed/retro_base.csv')
    retro_cnt = pd.read_csv('data/processed/retro_cnt.csv')

    # 1. Собираем статику (сумма контактов)
    total_contacts = retro_cnt.groupby('id', as_index=False)['cnt_contacts'].sum()
    total_contacts.rename(columns={'cnt_contacts': 'total_contacts'}, inplace=True)
    df_stat = retro_base.merge(total_contacts, on='id', how='left')
    df_stat['total_contacts'] = df_stat['total_contacts'].fillna(0).astype(int)

    # 2. Добавляем географию
    df_stat = add_geo_zones(df_stat)

    # 3. Запускаем поиск гипотез по тройкам признаков
    retro_segments = run_retro_grid_search(df_stat, n_factors=3, min_cars=50)
    retro_segments.to_csv('data/processed/final_retro_segments.csv', index=False)

Собираем Ретро-витрину...
Запуск Ретро-сканера гипотез (комбинации по 3 факторам, минимум 50 машин)...

ТОП-5 ЭКСКЛЮЗИВНЫХ АЛМАЗОВ (Высокий спрос):


,segment_name,median_contacts,cars_count,market_share_%
44,Москва (Город) | Классика (1980-1990) | С проб...,19.0,103,0.37
4,Москва (Город) | Классика (1980-1990) | Проект...,18.5,94,0.34
6,Москва (Город) | Янгтаймеры (1990-2000) | Энту...,16.0,210,0.75
81,Москва (Город) | Проект (<500к) | С пробегом (...,16.0,773,2.77
12,Питер (Город) | Янгтаймеры (1990-2000) | Проек...,15.0,742,2.66



ТОП-5 МУЗЕЙНЫХ ВИСЯКОВ (Долгая продажа):


,segment_name,median_contacts,cars_count,market_share_%
41,Дальний Восток | Классика (1980-1990) | С проб...,5.0,160,0.57
99,Сибирь | Энтузиаст (500к-1.5м) | С пробегом (>...,4.0,791,2.83
28,Сибирь | Янгтаймеры (1990-2000) | Энтузиаст (5...,4.0,777,2.78
2,Дальний Восток | Янгтаймеры (1990-2000) | Энту...,3.0,193,0.69
79,Дальний Восток | Энтузиаст (500к-1.5м) | С про...,3.0,222,0.79


In [4]:
def generate_retro_visualizations(
    base_path: str = 'data/processed/retro_base.csv',
    cnt_path: str = 'data/processed/retro_cnt.csv',
    output_dir: str = 'data/processed/'
) -> None:
    """
    Генерирует набор продуктовых графиков для ретро-сегмента.
    Анализирует влияние эпохи, географии и статуса 'Капсулы времени' на ликвидность.
    """
    print("Загрузка данных для визуализации...")
    # Читаем данные
    df_base = pd.read_csv(base_path)
    df_cnt = pd.read_csv(cnt_path)

    # Собираем статику (total_contacts) для базовых графиков
    total_contacts = df_cnt.groupby('id', as_index=False)['cnt_contacts'].sum()
    total_contacts.rename(columns={'cnt_contacts': 'total_contacts'}, inplace=True)
    df_stat = df_base.merge(total_contacts, on='id', how='left')
    df_stat['total_contacts'] = df_stat['total_contacts'].fillna(0).astype(int)

    # Настраиваем премиальный стиль графиков (темный фон смотрится элитно для ретро, но сделаем чистый whitegrid для презы)
    sns.set_theme(style="whitegrid", palette="deep")

    plt.figure(figsize=(10, 6))
    # Отсекаем выбросы больше 95 перцентиля для красивой отрисовки
    cap = df_stat['total_contacts'].quantile(0.95)
    plot_df = df_stat[df_stat['total_contacts'] <= cap]

    sns.boxplot(
        x='invest_tier', y='total_contacts', hue='is_time_capsule',
        data=plot_df, showfliers=False, palette=['#e74c3c', '#2ecc71']
    )
    plt.title('График 1: Влияние малого пробега (<50к) на ликвидность по классам авто', fontsize=14, pad=15)
    plt.xlabel('Инвестиционный статус (Цена)')
    plt.ylabel('Суммарные контакты')
    plt.legend(title='Капсула времени')
    plt.tight_layout()
    plt.savefig(f'{output_dir}retro_viz_1_timecapsule.png', dpi=300)
    plt.close()
    print(f"График 1 ({output_dir}retro_viz_1_timecapsule.png) сохранен: Эффект Капсулы времени")

    # Добавляем макро-зоны прямо здесь, если их нет в сохраненном CSV
    if 'geo_zone' not in df_stat.columns:
        moscow = ['Москва', 'Московская область']
        spb = ['Санкт-Петербург', 'Ленинградская область']
        south_caucasus = ['Краснодарский край', 'Ростовская область', 'Ставропольский край', 'Республика Дагестан', 'Чеченская Республика', 'Республика Северная Осетия — Алания']
        df_stat['geo_zone'] = np.where(df_stat['region'].isin(moscow), 'Москва и МО',
                              np.where(df_stat['region'].isin(spb), 'Питер и ЛО',
                              np.where(df_stat['region'].isin(south_caucasus), 'Юг и Кавказ', 'Регионы')))

    plt.figure(figsize=(12, 6))
    sns.barplot(
        x='geo_zone', y='total_contacts', hue='retro_era',
        data=df_stat, estimator=np.median, errorbar=None, palette='viridis'
    )
    plt.title('График 2: Где активнее всего ищут ретро-автомобили (Медиана)', fontsize=14, pad=15)
    plt.xlabel('Географическая зона')
    plt.ylabel('Медиана контактов')
    plt.legend(title='Эпоха выпуска')
    plt.tight_layout()
    plt.savefig(f'{output_dir}retro_viz_2_geography.png', dpi=300)
    plt.close()
    print(f"График 2 ({output_dir}retro_viz_2_geography.png) сохранен: География Эксклюзива")

    plt.figure(figsize=(8, 6))
    # Сводим таблицу: Эпоха vs Статус = Медиана звонков
    pivot_df = df_stat.pivot_table(
        values='total_contacts', index='retro_era', columns='invest_tier', aggfunc='median'
    )
    sns.heatmap(pivot_df, annot=True, cmap='YlOrRd', fmt=".1f", linewidths=.5)
    plt.title('График 3: Тепловая карта спроса на ретро-рынке', fontsize=14, pad=15)
    plt.xlabel('Инвестиционный статус')
    plt.ylabel('Эпоха выпуска')
    plt.tight_layout()
    plt.savefig(f'{output_dir}retro_viz_3_heatmap.png', dpi=300)
    plt.close()
    print(f"График 3 ({output_dir}retro_viz_2_geography.png) сохранен: Матрица спроса (Heatmap)")

    # Для этого графика мержим логи и базу по дням
    df_dyn = df_cnt.merge(df_base[['id', 'retro_era']], on='id', how='inner')

    plt.figure(figsize=(12, 6))
    # Считаем медиану звонков по каждому дню для каждой эпохи (берем первые 30 дней)
    avg_curves = df_dyn[df_dyn['day'] <= 30].groupby(['retro_era', 'day'])['cnt_contacts'].median().reset_index()

    sns.lineplot(x='day', y='cnt_contacts', hue='retro_era', data=avg_curves, marker='o', linewidth=2)
    plt.title('График 4: Плоские кривые спроса на ретро-автомобили (до 30 дней)', fontsize=14, pad=15)
    plt.xlabel('День жизни объявления')
    plt.ylabel('Медианное кол-во контактов')
    plt.xticks(range(1, 31, 2)) # Шаг в 2 дня для красоты
    plt.tight_layout()
    plt.savefig(f'{output_dir}retro_viz_4_time_curves.png', dpi=300)
    plt.close()
    print(f"График 4 ({output_dir}retro_viz_2_geography.png) сохранен: Кривые затухания спроса")

if __name__ == "__main__":
    generate_retro_visualizations()

Загрузка данных для визуализации...
График 1 (data/processed/retro_viz_1_timecapsule.png) сохранен: Эффект Капсулы времени
График 2 (data/processed/retro_viz_2_geography.png) сохранен: География Эксклюзива
График 3 (data/processed/retro_viz_2_geography.png) сохранен: Матрица спроса (Heatmap)
График 4 (data/processed/retro_viz_2_geography.png) сохранен: Кривые затухания спроса
